# LPPL Bubble Detection Analysis

Log-Periodic Power Law (LPPL) model for detecting financial bubbles and crashes.

**Pipeline:**
1. Load price data (OHLCV CSV)
2. Fit LPPL model to the full price series
3. Run nested-window analysis to get rolling bubble confidence scores
4. Export enhanced CSV with `bubble_pos_conf` and `bubble_neg_conf` columns

## 1. Setup

In [ ]:
# Install lppls if not already present
import importlib, subprocess, sys
if importlib.util.find_spec('lppls') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lppls'])

In [ ]:
from lppls import lppls
import numpy as np
import pandas as pd
import re
import os
from datetime import datetime as dt
from matplotlib import pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 2. Configuration

Set `DATA_FILE` to the path of your OHLCV CSV. The file must have a `date` or `dateTime` column in `DD/MM/YYYY` or `DD/MM/YYYY HH:MM` format.

In [ ]:
DATA_FILE = '../data/NVDA_.csv'   # <-- change this to use a different file
MAX_SEARCHES = 25                  # LPPL optimizer restarts per fit
WINDOW_SIZE = 120                  # days per rolling window
SMALLEST_WINDOW = 30
WORKERS = 8                        # parallel processes for nested fits

## 3. Load & Parse Data

In [ ]:
data = pd.read_csv(DATA_FILE)

def check_date_format(s):
    return bool(re.match(r'^\d{2}/\d{2}/\d{4}$', s))

def check_datetime_format(s):
    return bool(re.match(r'^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$', s))

date_col = None
if 'date' in data.columns:
    date_col = 'date'
    if data['date'].apply(check_date_format).all():
        data['date'] = pd.to_datetime(data['date'], format='%d/%m/%Y').dt.strftime('%Y-%m-%d')
    time = [pd.Timestamp.toordinal(dt.strptime(t, '%Y-%m-%d')) for t in data['date']]
elif 'dateTime' in data.columns:
    date_col = 'dateTime'
    if data['dateTime'].apply(check_datetime_format).all():
        data['dateTime'] = pd.to_datetime(data['dateTime'], format='%d/%m/%Y %H:%M').dt.strftime('%Y-%m-%d %H:%M')
    time = [pd.Timestamp.toordinal(dt.strptime(t.split()[0], '%Y-%m-%d')) for t in data['dateTime']]
else:
    raise ValueError("CSV must contain a 'date' or 'dateTime' column")

print(f"Loaded {len(data)} rows | date column: '{date_col}'")
data.head()

## 4. LPPL Model Fit

Fits the LPPL equation to the full log-price series and plots the result.

In [ ]:
price = np.log(data['close'].values)
observations = np.array([time, price])

lppls_model = lppls.LPPLS(observations=observations)
tc, m, w, a, b, c, c1, c2, O, D = lppls_model.fit(MAX_SEARCHES)

print(f"tc={tc:.2f}  m={m:.3f}  ω={w:.3f}  A={a:.3f}  B={b:.3f}")

In [ ]:
time_ord = [pd.Timestamp.fromordinal(d) for d in lppls_model.observations[0, :].astype('int32')]
t_obs = lppls_model.observations[0, :]
lppls_fit = [lppls_model.lppls(t, tc, m, w, a, b, c1, c2) for t in t_obs]
log_price = lppls_model.observations[1, :]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_ord, log_price, label='log(price)', color='black', linewidth=0.75)
ax.plot(time_ord, lppls_fit, label='LPPL fit', color='royalblue', alpha=0.7, linewidth=1.5)
ax.set_ylabel('ln(p)')
ax.set_title('LPPL Fit vs Log Price')
ax.legend()
ax.grid(linestyle='--', alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Nested-Window Analysis (Rolling Bubble Detection)

Slides a `WINDOW_SIZE`-day window across the series, fitting LPPL at each position.
Produces `pos_conf` (upward bubble) and `neg_conf` (downward bubble) confidence scores.

> **Note:** This step takes ~20–30 minutes on a typical dataset. Progress is shown below.

In [ ]:
res = lppls_model.mp_compute_nested_fits(
    workers=WORKERS,
    window_size=WINDOW_SIZE,
    smallest_window_size=SMALLEST_WINDOW,
    outer_increment=1,
    inner_increment=5,
    max_searches=MAX_SEARCHES
)

## 6. Results — Bubble Indicators

In [ ]:
res_df = lppls_model.compute_indicators(res)

ord_ts = res_df['time'].astype('int32')
ts = [pd.Timestamp.fromordinal(d) for d in ord_ts]

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(16, 9))

# Positive bubble
ax1r = ax1.twinx()
ax1.plot(ts, res_df['price'], color='black', linewidth=0.75, label='log(price)')
ax1r.plot(ts, res_df['pos_conf'], color='red', alpha=0.6, label='pos bubble confidence')
ax1.set_ylabel('ln(p)')
ax1r.set_ylabel('Positive bubble confidence')
ax1r.legend(loc=2)
ax1.grid(linestyle='--', alpha=0.5)
ax1.set_title('Upward Bubble Indicator')

# Negative bubble
ax2r = ax2.twinx()
ax2.plot(ts, res_df['price'], color='black', linewidth=0.75)
ax2r.plot(ts, res_df['neg_conf'], color='green', alpha=0.6, label='neg bubble confidence')
ax2.set_ylabel('ln(p)')
ax2r.set_ylabel('Negative bubble confidence')
ax2r.legend(loc=2)
ax2.grid(linestyle='--', alpha=0.5)
ax2.set_title('Downward Bubble Indicator')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Export Enhanced CSV

Appends `bubble_pos_conf` and `bubble_neg_conf` to the original data and saves to `../data/`.

In [ ]:
row_diff = len(data) - len(res_df)
pad = [0.0] * row_diff

data['bubble_pos_conf'] = pad + list(res_df['pos_conf'].values)
data['bubble_neg_conf'] = pad + list(res_df['neg_conf'].values)
data.set_index(date_col, inplace=True)

base_name = os.path.splitext(os.path.basename(DATA_FILE))[0]
out_path = f'../data/output_{base_name}.csv'
data.to_csv(out_path)

print(f"Saved: {out_path}")
data.tail()